# Stretch: the Spotify route, worked solutions

The answers to the stretch notebook, on the practice export. Try each one
before looking.

The field names are the ones that appear in real extended streaming
history files. The values are made up. Check the names against the *Read
Me First* file that comes with your own export.

In [ ]:
import os
from pathlib import Path

import pandas as pd

here = Path.cwd()
while not (here / "data" / "clean" / "plays.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)
print("working from:", Path.cwd())

In [ ]:
# Run this cell as it is. It writes a small PRACTICE export: the field names
# are the real ones, every value is made up. Nothing here is anyone's data.
import json
import random
from datetime import datetime, timedelta, timezone

PRACTICE = Path("output") / "session-11" / "spotify-practice"
PRACTICE.mkdir(parents=True, exist_ok=True)

rng = random.Random(11)
artists = {
    "Glass Tram": ["Broken Lines", "Amber Ferry", "Night Bus"],
    "Sara Lindqvist": ["Salt Lines", "Small Hours", "Harbour"],
    "DJ Kompas": ["Broken Lines", "North Star"],
    "Fjord & Flint": ["Golden Garden", "Summer Season"],
    "Odessa Brass": ["Golden Avenue"],
}
shows = {"The Long Listen": ["Episode 12", "Episode 13"]}
start = datetime(2024, 1, 1, tzinfo=timezone.utc)


def practice_record():
    """One made-up stream, with the field names of a real extended history file."""
    ended = start + timedelta(seconds=rng.randrange(2 * 365 * 24 * 3600))
    is_podcast = rng.random() < 0.08
    played_ms = rng.choice([rng.randrange(1_000, 30_000), rng.randrange(60_000, 330_000)])
    record = {
        "ts": ended.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "platform": rng.choice(["android", "ios", "windows"]),
        "ms_played": played_ms if not is_podcast else rng.randrange(300_000, 3_000_000),
        "conn_country": rng.choice(["NL", "NL", "NL", "DE"]),
        "master_metadata_track_name": None,
        "master_metadata_album_artist_name": None,
        "master_metadata_album_album_name": None,
        "spotify_track_uri": None,
        "episode_name": None,
        "episode_show_name": None,
        "spotify_episode_uri": None,
        "reason_start": rng.choice(["trackdone", "clickrow", "fwdbtn"]),
        "reason_end": rng.choice(["trackdone", "fwdbtn", "endplay"]),
        "shuffle": rng.random() < 0.5,
        "skipped": rng.choice([True, False, False, False, None]),
        "offline": False,
        "offline_timestamp": None,
        "incognito_mode": False,
    }
    if is_podcast:
        show = rng.choice(list(shows))
        record["episode_name"] = rng.choice(shows[show])
        record["episode_show_name"] = show
        record["spotify_episode_uri"] = "spotify:episode:practice"
    else:
        artist = rng.choice(list(artists))
        record["master_metadata_track_name"] = rng.choice(artists[artist])
        record["master_metadata_album_artist_name"] = artist
        record["master_metadata_album_album_name"] = "Practice Album"
        record["spotify_track_uri"] = "spotify:track:practice"
    return record


records = sorted((practice_record() for _ in range(600)), key=lambda r: r["ts"])
files = {
    "Streaming_History_Audio_2024_0.json": records[:350],
    "Streaming_History_Audio_2024-2025_1.json": records[350:],
}
for name, chunk in files.items():
    with open(PRACTICE / name, "w", encoding="utf-8") as f:
        json.dump(chunk, f, indent=1)
    print(f"wrote {PRACTICE / name}  ({len(chunk)} streams)")

## Stretch 1: a JSON file is a list of dictionaries

In [ ]:
first_file = PRACTICE / "Streaming_History_Audio_2024_0.json"

with open(first_file, encoding="utf-8") as f:
    records = json.load(f)

print(type(records), len(records))
print(records[0])

assert len(records) == 350
assert records[0]["ts"] == "2024-01-03T14:21:38Z"

A list of 350 dictionaries, and `json.load` turned JSON's `null` into
Python's `None` and `false` into `False` on the way in. You could answer
questions with session 4 loops from here. But this is exactly the shape a
DataFrame is built from:

In [ ]:
one = pd.read_json(first_file)

print(len(one))
print(one.dtypes)

assert len(one) == len(records)

Two columns to notice:

* `ts` is `object`: text. pandas only guesses that a column holds dates
  when its name looks like a date (`..._at`, `date`, `timestamp` and a few
  others), and `ts` does not. Stretch 3 converts it on purpose.
* `skipped` is `float64`, not `bool`. Some records have `null` there, and
  a column of `True`, `False` and missing becomes `1.0`, `0.0` and `NaN`.
  Stretch 5 has to deal with that.

## Stretch 2: all the files at once

In [ ]:
def load_streams(folder):
    """Return every Streaming_History_Audio file in folder as one DataFrame, sorted by ts."""
    paths = sorted(Path(folder).glob("Streaming_History_Audio_*.json"))
    for path in paths:
        print("  reading", path.name)
    frames = [pd.read_json(path) for path in paths]
    streams = pd.concat(frames, ignore_index=True)
    return streams.sort_values("ts", ignore_index=True)


streams = load_streams(PRACTICE)
print(len(streams), "streams")

assert len(streams) == 600

The `2024-2025_1` file is read **before** the `2024_0` file. Sorting file
names compares them character by character, and `-` comes before `_`, so
the "second" file sorts first.

Here that is harmless, because `load_streams` sorts the rows by `ts`
afterwards, and `ts` is written `YYYY-MM-DDTHH:MM:SSZ`: the session 4 rule
that a date written biggest-part-first sorts correctly as text. That is
the safe habit. **File names are for people; the order of the data should
come from the data.**

## Stretch 3: what time was it, really?

In [ ]:
streams["ts"] = pd.to_datetime(streams["ts"], utc=True)
streams["local"] = streams["ts"].dt.tz_convert("Europe/Amsterdam")

changed_date = (streams["ts"].dt.date != streams["local"].dt.date).sum()
print(f"{changed_date} of {len(streams)} streams have a different date in Amsterdam")

streams.loc[streams["ts"].dt.date != streams["local"].dt.date, ["ts", "local"]].head(3)

In [ ]:
assert changed_date == 35

Amsterdam is one hour ahead of UTC in winter and two in summer, so a
stream that ended at 23:30 UTC ended after midnight in Amsterdam, on the
next day. 35 of 600 here.

* **Minutes per year:** barely matters. Only the streams around midnight
  on New Year's Eve can change year.
* **Time of day, or day of the week:** matters a lot. In UTC your evening
  listening is an hour or two early, and late-night streams land on the
  wrong day. Convert first.

And remember `ts` is when the stream *ended*. For a four-minute song that
makes no difference. For a two-hour podcast episode, "what time did I
start listening" is two hours earlier than `ts`.

## Stretch 4: music or podcast?

In [ ]:
music = streams[streams["master_metadata_track_name"].notna()]
podcasts = streams[streams["episode_name"].notna()]

print(f"music:    {len(music):>4} streams  {music['ms_played'].sum() / 60_000:>7.1f} minutes")
print(f"podcasts: {len(podcasts):>4} streams  {podcasts['ms_played'].sum() / 60_000:>7.1f} minutes")

assert (len(music), len(podcasts)) == (556, 44)
assert len(music) + len(podcasts) == len(streams)    # every stream is one or the other

44 podcast streams carry more minutes (1,200.8) than 556 music streams
(1,004.9), because an episode lasts half an hour and a song four
minutes. In the practice file that is exaggerated, but the effect is real:
a "minutes per artist" project that forgot to filter would find podcast
rows with no artist at all, and a total for the year that was mostly
talking. Decide at the start which one your project is about.

## Stretch 5: what counts as a play?

In [ ]:
artist = "master_metadata_album_artist_name"

all_streams = music[artist].value_counts()
real_plays = music[music["ms_played"] >= 30_000][artist].value_counts()

print("counting every stream:")
print(all_streams.head(3))
print()
print("counting streams of 30 seconds or more:")
print(real_plays.head(3))

top_all = all_streams.idxmax()
top_30s = real_plays.idxmax()

assert top_all == "Glass Tram"
assert top_30s == "DJ Kompas"

In [ ]:
print(streams["skipped"].value_counts(dropna=False))

assert streams["skipped"].isna().sum() == 123

Counting everything, Glass Tram is the top artist with 120 streams.
Counting only streams of 30 seconds or more, DJ Kompas is, with 63. Same
file, same question, different winner, and the only change was a
definition. That is session 9's lesson again: every decision changes the
answer.

Thirty seconds is a common choice, because it is roughly "listened rather
than skipped past", and it is easy to say out loud: *"a play is a stream
of at least thirty seconds"*. The honest presentation states the
threshold, and ideally says whether the answer changes without it.

`skipped` is missing for 123 of the 600 practice streams. Treating missing
as "not skipped" is one choice; leaving those rows out of any skip-rate
question is another, and probably the more honest one for a skip rate.
What you must not do is let pandas decide quietly: `mean()` skips `NaN`
without telling you.

## Stretch 6: into the shape of the course data

In [ ]:
def spotify_to_plays(streams, min_ms=30_000):
    """Return music streams of at least min_ms as rows shaped like data/clean/plays.csv."""
    music = streams[streams["master_metadata_track_name"].notna()]
    music = music[music["ms_played"] >= min_ms]
    local = pd.to_datetime(music["ts"], utc=True).dt.tz_convert("Europe/Amsterdam")
    return pd.DataFrame({
        "played_at": local.dt.strftime("%Y-%m-%d"),
        "artist_name": music["master_metadata_album_artist_name"],
        "track_name": music["master_metadata_track_name"],
        "minutes_played": (music["ms_played"] / 60_000).round(2),
        "device": music["platform"],
        # Missing means Spotify did not record it. Counted as not skipped: a decision.
        "skipped": music["skipped"].fillna(0).astype(int),
    })


plays = spotify_to_plays(streams)
print(plays.head())

course_columns = pd.read_csv("data/clean/plays.csv", nrows=1).columns
assert set(plays.columns) <= set(course_columns)
assert len(plays) == 284

Every column name is one the course data already uses, so the session 10
pipeline's analysis code has a fighting chance of working unchanged.
`pd.to_datetime(..., utc=True)` is used again inside the function so that
it works whether or not `ts` has already been converted: a function that
depends on what you happened to run earlier in the notebook is a function
that will break in the script.

Now the worked example's question.

In [ ]:
missing = [c for c in ["genre", "country"] if c not in plays.columns]
print("columns the course data has and Spotify does not:", missing)

assert missing == ["genre", "country"]

There is no genre. Nothing in the extended streaming history says what
genre a track is, and nothing says where an artist is from
(`conn_country` is where **you** were when you listened). Spotify's list
of what is in the export confirms it.

That is a scoping fact, and it is much better to find it now than in
week two: **a Spotify project asks about artists, tracks, time and
devices, not genres.** Getting genres means Spotify's Web API, which needs
an account, keys and a lot of requests. That is a "won't, this time" item.

The same question per artist works:

In [ ]:
plays["year"] = plays["played_at"].str[:4]

per_artist = plays.groupby(["artist_name", "year"])["minutes_played"].sum().unstack("year").round(1)
per_artist["change"] = (per_artist["2025"] - per_artist["2024"]).round(1)
per_artist = per_artist.sort_values("change")
per_artist

In [ ]:
assert per_artist["change"].idxmax() == "DJ Kompas"
assert per_artist["change"].idxmin() == "Sara Lindqvist"

Which artist grew and which shrank in the practice data means nothing,
because the values are random. What matters is that the whole route works:
from a folder of JSON files to a table in the shape of notebook 01, in
about forty lines. When the real export arrives, that is the head start.

## When your real export arrives

1. Unzip it into `project/data/spotify/`. Then read the private data slide
   **before** your next `git add`: the export contains IP addresses and
   other things that do not belong in a public repository.
2. Open the *Read Me First* file and compare its field names with the ones
   used here. If one differs, change it here; the logic stays the same.
3. Point `load_streams` at the folder, run `first_look` from notebook 01 on
   the result, and look hard at `platform`, `reason_end` and `skipped`
   with `value_counts(dropna=False)`.
4. `load_streams` and `spotify_to_plays` are the `load` and the first half
   of `clean` for `project/pipeline.py`.